# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset is described via a Croissant schema at the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```
- **Title**: Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
- **Identifier**: 10.71728/senscience.y7m0-f273


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and preview the main information.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (FAIR² dataset)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# The metadata object provides rich information about the dataset
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Let's review what record sets are available in this Croissant dataset, and explore the fields (columns) and their unique `@id` identifiers.

This is important, because you'll reference all fields and records by their `@id`.

In [ ]:
# List all record sets with names and @id
print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"- {rs.name} (@id: {rs.id})")

# For each record set, list fields and their @id
for rs in metadata.record_sets:
    print(f"\nFields for record set: {rs.name} (@id: {rs.id}):")
    for fld in rs.fields:
        print(f"  - {fld.name} (@id: {fld.id}) [dataType: {getattr(fld, 'data_type', 'N/A')}]" )

## 3. Data Extraction
Load data for each record set into pandas DataFrames for analysis (using the record set `@id`).

**Note:** Please refer to the output above for the correct `@id` values of record sets.

In [ ]:
# Collect all record set @id's
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    # `records()` yields dicts keyed by field @id
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set '{record_set_id}'. Columns: {list(df.columns)}\n")

# List one record set for working in EDA
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Example record set to explore: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform exploratory analysis on a selected DataFrame using field `@id`s. We'll demonstrate filtering numeric fields, normalizing values, and grouping by categorical fields where possible.

*Adjust the example below to match actual available field IDs in your dataset.*

In [ ]:
# Choose the record set / DataFrame and relevant fields by their @id
df = dataframes[main_record_set_id]

# List all available columns/@id (choose appropriate for EDA)
print("Columns available:")
print(df.columns.tolist())

# Example: Pick a numeric field and a group field by inspecting columns above
numeric_field_id = None
group_field_id = None

# Attempt to find a likely numeric and group field by heuristics on id/name
for col in df.columns:
    if ("log_likelihood" in col.lower()) or ("coefficient" in col.lower()) or ("p_value" in col.lower()):
        numeric_field_id = col
    if ("county" in col.lower()) or ("ward" in col.lower()) or ("gender" in col.lower()) or ("group" in col.lower()) or ("class" in col.lower()):
        group_field_id = col
    if numeric_field_id and group_field_id:
        break

# Fallback if not found
if not numeric_field_id:
    numeric_field_id = df.select_dtypes('number').columns[0] if not df.select_dtypes('number').empty else df.columns[0]
if not group_field_id:
    if len(df.columns) > 1:
        group_field_id = df.columns[1]
    else:
        group_field_id = df.columns[0]

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Ensure numeric (try to convert if not)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (total: {len(filtered_df)}):")
display(filtered_df.head())

# Add normalized field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' (first 5 rows):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the group field, compute mean of numeric field if possible
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped means of '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df.head())
else:
    print(f"Group field '{group_field_id}' not found in columns.")

## 5. Visualization
Let's visualize the normalized numeric field and explore distributions or group relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of normalized numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=20, kde=True, color='teal')
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Count")
plt.show()

# If group_field_id available, boxplot by group
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded the FAIR² dataset defined by a Croissant schema via `mlcroissant` and explored its record set structure and field `@id`s.
- We extracted the records into pandas DataFrames, performed basic filtering and normalization of a numeric field using its `@id`, and grouped data by a key attribute.
- Visualization revealed summary statistics and group distributions for further analysis.

For further research, use the field and record set `@id`s for reproducible querying and robust data extraction.